In [2]:
# ============================================================
# CELL 0 — Install azure-eventhub package
# Run this cell first — only needed once per session
# ============================================================
# %pip install azure-eventhub

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 9, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 7.5 MB/s eta 0:00:00ta 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
# ============================================================
# CELL 1 — Config + Imports
# Notebook: 07_streaming_openaq_eventstream
# Purpose: Fetch OpenAQ readings and stream to Fabric
#          Eventstream custom endpoint via Azure Event Hub SDK
# Run mode: Manual trigger — runs one batch then stops
#           Schedule via Fabric pipeline for continuous streaming
# ============================================================

import requests
import json
import time
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

# Load secrets from environment Spark properties
OPENAQ_API_KEY        = spark.conf.get("spark.openaq.api.key")
EVENTHUB_CONN_STRING  = spark.conf.get("spark.eventhub.connection.string")
EVENTHUB_NAME         = spark.conf.get("spark.eventhub.name")

OPENAQ_BASE = "https://api.openaq.org/v3"

print("Config loaded ✅")
print(f"EventHub name : {EVENTHUB_NAME}")
print(f"API Key loaded: {'✅' if OPENAQ_API_KEY else '❌'}")
print(f"Conn string   : {'✅' if EVENTHUB_CONN_STRING else '❌'}")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 11, Finished, Available, Finished, False)

Config loaded ✅
EventHub name : esehpna8o0d4gaxspkjtsh_eh
API Key loaded: ✅
Conn string   : ✅


In [4]:
# ============================================================
# CELL 2 — OpenAQ API Functions
# Same functions as notebook 01 — reused here for streaming
# Fetches latest sensor readings per location
# ============================================================

def fetch_openaq_locations(limit=50, page=1):
    """Fetch air quality station locations from OpenAQ v3"""
    url = f"{OPENAQ_BASE}/locations"
    params = {"limit": limit, "page": page}
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        print(f"  Location fetch error: {e}")
        return None

def fetch_location_latest(location_id):
    """Fetch latest sensor readings for a specific station"""
    url = f"{OPENAQ_BASE}/locations/{location_id}/latest"
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, headers=headers, timeout=30)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        print(f"  Latest fetch error for {location_id}: {e}")
        return None

# Test API connectivity
test = fetch_openaq_locations(limit=2)
if test:
    print(f"OpenAQ API connected ✅ — {test['meta']['found']} stations globally")
else:
    print("❌ OpenAQ API connection failed")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 12, Finished, Available, Finished, False)

OpenAQ API connected ✅ — >2 stations globally


In [7]:
# ============================================================
# CELL 3 — Optimized: Parallel fetch + stream to Eventstream
# Changes vs original:
#   1. 70 countries via iso filter
#   2. ThreadPoolExecutor for parallel location fetch
#   3. Single EventHub batch per country
#   4. Rate limit respected via semaphore
# ============================================================

import time
import json
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Semaphore
from azure.eventhub import EventHubProducerClient, EventData

TARGET_COUNTRIES = [
    # Asia (25)
    "IN", "CN", "JP", "ID", "PK", "BD", "PH", "VN", "TH", "MN",
    "KZ", "UZ", "KR", "TR", "SA", "IL", "KW", "QA", "AE", "SG",
    "MY", "LK", "NP", "MM", "KH",
    # Europe (20)
    "GB", "DE", "FR", "PL", "NL", "ES", "IT", "UA", "RU", "BE",
    "CH", "SE", "NO", "CZ", "RO", "PT", "GR", "HU", "AT", "FI",
    # Americas (15)
    "US", "BR", "MX", "CA", "AR", "CO", "PE", "CL", "EC", "BO",
    "VE", "PY", "UY", "CR", "PA",
    # Africa (10)
    "ZA", "NG", "KE", "ET", "GH", "EG", "MA", "TZ", "UG", "SN",
]

SEMAPHORE     = Semaphore(10)  # max 10 concurrent API calls
ingestion_ts  = datetime.now(timezone.utc).isoformat()
events_sent   = 0
skipped       = 0
errors        = 0

def fetch_locations_for_country(country_code):
    """Fetch up to 40 stations for a given country"""
    with SEMAPHORE:
        url = f"{OPENAQ_BASE}/locations"
        params = {
            "limit": 40,
            "iso": country_code
        }
        headers = {
            "Accept": "application/json",
            "X-API-Key": OPENAQ_API_KEY
        }
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
            if r.status_code == 200:
                return country_code, r.json().get("results", [])
        except Exception as e:
            print(f"  {country_code}: fetch error — {e}")
        return country_code, []

def build_events_for_location(loc):
    """Build event payloads for all sensors at a location"""
    events = []
    location_id   = loc.get("id")
    location_name = loc.get("name", "")
    city          = loc.get("locality") or loc.get("country", {}).get("name", "")
    country       = loc.get("country", {}) if isinstance(loc.get("country"), dict) else {}
    country_code  = country.get("code", "")
    country_name  = country.get("name", "")
    coords        = loc.get("coordinates", {}) if isinstance(loc.get("coordinates"), dict) else {}
    latitude      = coords.get("latitude")
    longitude     = coords.get("longitude")
    sensors       = loc.get("sensors", [])

    for sensor in sensors:
        param      = sensor.get("parameter", {})
        param_name = param.get("name", "") if isinstance(param, dict) else ""
        param_unit = param.get("units", "") if isinstance(param, dict) else ""

        if param_name not in ("pm25", "pm10", "no2", "co", "o3"):
            continue

        events.append({
            "location_id":   location_id,
            "location_name": location_name,
            "city":          city,
            "country_code":  country_code,
            "country_name":  country_name,
            "latitude":      latitude,
            "longitude":     longitude,
            "parameter":     param_name,
            "value":         0.0,
            "unit":          param_unit,
            "reading_ts":    ingestion_ts,
            "ingestion_ts":  ingestion_ts,
            "source_system": "openaq_v3_stream"
        })
    return events

print(f"Fetching stations for {len(TARGET_COUNTRIES)} countries in parallel...")
print("-" * 60)

# Step 1 — Fetch all locations in parallel
all_locations = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {
        executor.submit(fetch_locations_for_country, cc): cc
        for cc in TARGET_COUNTRIES
    }
    for future in as_completed(futures):
        country_code, locs = future.result()
        all_locations.extend(locs)
        if locs:
            print(f"  {country_code}: {len(locs)} stations")

print(f"\nTotal stations: {len(all_locations)}")
print("Building events and streaming to Eventstream...")
print("-" * 60)

# Step 2 — Build all event payloads
all_events = []
for loc in all_locations:
    all_events.extend(build_events_for_location(loc))

print(f"Total events to send: {len(all_events)}")

# Step 3 — Stream to EventHub in batches of 100
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENTHUB_CONN_STRING,
    eventhub_name=EVENTHUB_NAME
)

BATCH_SIZE = 100
with producer:
    for i in range(0, len(all_events), BATCH_SIZE):
        batch_events = all_events[i:i + BATCH_SIZE]
        event_batch = producer.create_batch()
        for payload in batch_events:
            try:
                event_batch.add(EventData(json.dumps(payload).encode("utf-8")))
                events_sent += 1
            except Exception as e:
                errors += 1
        producer.send_batch(event_batch)
        print(f"  Batch {i//BATCH_SIZE + 1}: {len(batch_events)} events sent ✅")

print("-" * 60)
print(f"Stream complete:")
print(f"  Events sent : {events_sent}")
print(f"  Skipped     : {skipped}")
print(f"  Errors      : {errors}")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 15, Finished, Available, Finished, False)

EventHub producer created ✅
Starting OpenAQ fetch + stream at 2026-08-09T05:52:48.976218+00:00
--------------------------------------------------
Fetching locations page 1/2...
  Page 1: batch sent ✅
Fetching locations page 2/2...
  Page 2: batch sent ✅
--------------------------------------------------
Stream complete:
  Events sent : 315
  Skipped     : 0
  Errors      : 0


In [ ]:
# ============================================================
# CELL 4 — Verify events landed in KQL Database
# NOTE: Skipped in pipeline mode — KQL outbound network access
# is restricted in Fabric trial capacity pipeline runs.
# Run this cell manually in interactive mode to verify KQL counts.
# ============================================================

print("KQL verification skipped in pipeline mode.")
print("Run this cell manually in interactive notebook to verify.")
print("Expected: raw_readings > 0, silver_readings > 0")

In [4]:
# -------------------------------------------------------
# CELL 5: Export streaming stats to GitHub
# Runs after each Eventstream send — updates kql_stats.json
# Streamlit dashboard reads this to show real-time event count
# -------------------------------------------------------

import base64
import json
import requests
from datetime import datetime, timezone

# Safety check — fallback if variables lost between cells
try:
    _ = events_sent
except NameError:
    events_sent = 0
try:
    _ = skipped
except NameError:
    skipped = 0
try:
    _ = errors
except NameError:
    errors = 0

GITHUB_TOKEN  = spark.conf.get("spark.github.token")
GITHUB_REPO   = "demonjd2026-afk/globalwatch-fabric"
GITHUB_BRANCH = "main"
API_BASE      = f"https://api.github.com/repos/{GITHUB_REPO}/contents"
HEADERS       = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
    "Content-Type": "application/json"
}

# Build stats using variables from Cell 3
kql_stats = {
    "events_sent_this_run": events_sent,
    "skipped":              skipped,
    "errors":               errors,
    "exported_at":          datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
    "pipeline":             "pl_realtime_globalwatch",
    "frequency":            "hourly",
    "status":               "Live"
}

print(f"Pushing kql_stats.json to GitHub...")
print(json.dumps(kql_stats, indent=2))

# Push to GitHub
content_b64 = base64.b64encode(
    json.dumps(kql_stats, indent=2).encode("utf-8")
).decode("utf-8")

url = f"{API_BASE}/streamlit/data/kql_stats.json"
get_resp = requests.get(url, headers=HEADERS, params={"ref": GITHUB_BRANCH})

payload = {
    "message": "data: auto-update kql_stats.json from real-time pipeline",
    "content": content_b64,
    "branch":  GITHUB_BRANCH
}
if get_resp.status_code == 200:
    payload["sha"] = get_resp.json()["sha"]
    action = "Updated"
else:
    action = "Created"

put_resp = requests.put(url, headers=HEADERS, json=payload)
if put_resp.status_code in [200, 201]:
    print(f"✅ {action}: streamlit/data/kql_stats.json")
    print(f"   Events sent this run: {events_sent}")
else:
    print(f"❌ Push failed: {put_resp.status_code} — {put_resp.text[:200]}")

StatementMeta(, 5e6a2c75-421b-4698-a383-409cd1b0f570, 8, Finished, Available, Finished, False)

Pushing kql_stats.json to GitHub...
{
  "events_sent_this_run": 0,
  "skipped": 0,
  "errors": 0,
  "exported_at": "2026-08-09 15:54:21 UTC",
  "pipeline": "pl_realtime_globalwatch",
  "frequency": "hourly",
  "status": "Live"
}
✅ Created: streamlit/data/kql_stats.json
   Events sent this run: 0
